In [2]:
import warnings
warnings.filterwarnings('ignore')

Vanishing/Exploding Gradients Problems

Glorot and He Initialization

this is variance<br><br><br>
Glorot    ---  None, Tanh, Logistic, Softmax   ---    1 / fanavg<br>
He          ----    ReLU & variants         ---         2 / fanin<br>
LeCun        ---       SELU                ---         1 / fanin

By default, Keras uses Glorot initialization with a uniform distribution. You can change this to He initialization by setting kernel_initializer="he_uniform" or kernel_initializer="he_normal" when creating a layer, like this:

In [3]:
from tensorflow import keras
keras.layers.Dense(10,activation='relu',kernel_initializer='he_normal')

<Dense name=dense, built=False>

If you want He initialization with a uniform distribution, but based on fanavg rather
than fanin, you can use the VarianceScaling initializer like this:

In [4]:
he_avg_init = keras.initializers.VarianceScaling(scale=2,mode='fan_avg',
                                                 distribution='uniform')
keras.layers.Dense(10,activation='sigmoid',kernel_initializer=he_avg_init)

<Dense name=dense_1, built=False>

Nonsaturating Activation Functions

 LeakyReLUα
(z) = max(αz, z) 

 exponential linear unit (ELU)

ELU(z)   a(exp(z)-1)  if z<0  ,   z if z>=0

SELU(Scaled ELU)

To use the leaky ReLU activation function, you must create a LeakyReLU instance like this:

In [5]:
leaky_relu = keras.layers.LeakyReLU(alpha=0.2)
layer = keras.layers.Dense(10,activation=leaky_relu,
                           kernel_initializer='he_normal')

Batch Normalization

Implementing Batch Normalization with keras

In [6]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28,28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(30,activation='elu',kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(40,activation='elu',kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10,activation='softmax')
])

In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 784)            │         3,136 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30)             │        23,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30)             │           120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 40)             │         1,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 40)             │           160 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           410 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,616 (111.78 KB)

 Trainable params: 26,908 (105.11 KB)

 Non-trainable params: 1,708 (6.67 KB)

In [8]:
[(var.name, var.trainable) for var in model.layers[1].variables]

[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

In [9]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28,28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(30,kernel_initializer='he_normal',use_bias=False),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('elu'),
    keras.layers.Dense(20,activation='softmax')
])

Why Bias to False?<br> Because BatchNormalization just applies weight and bias to it.

<big>Gradient Clipping</big>

Another popular technique to lessen the exploding gradients problem is to simply
clip the gradients during backpropagation so that they never exceed some threshold.

It handles Only the gradient exploding problem---How I understood the concept

Mostly used in RNNs

In [10]:
optimizer = keras.optimizers.SGD(clipvalue=1.0)

Reusing Pretrained Layers

Transfer Learning with keras

TO load a Model: model_A  = keras.models.load_model("my_model_A.h5)

model_B_on_A = keras.models.Sequential(model_A.layers[-1])

Note that model_A and model_B_on_A now share some layers. When you train
model_B_on_A, it will also affect model_A. If you want to avoid that, you need to clone
model_A before you reuse its layers. To do this, you must clone model A’s architecture,
then copy its weights (since clone_model() does not clone the weights):

model_A_clone = keras.models.clone(model_A)<br>
model_A_clone = set_weights(model_A.get_weights())

<big>Unsupervised Pretraining</big>

Autoencoders and RBMs

<big>Pretraining on an Auxiliary Task</big>

<big>Faster Optimizers</big>

Momentum Optimization

Recall that Gradient Descent simply updates the weights θ by directly subtracting the
gradient of the cost function J(θ) with regards to the weights (∇θ
J(θ)) multiplied by
the learning rate η. The equation is: θ ← θ – η∇θ
J(θ). It does not care about what the
earlier gradients were. If the local gradient is tiny, it goes very slowly.<br>
Momentum optimization cares a great deal about what previous gradients were: at
each iteration, it subtracts the local gradient from the momentum vector m (multiplied by the learning rate η), and it updates the weights by simply adding this
momentum vector 

m -> βm − η∇θJ(θ)

Useful to avoid local minimas

In [12]:
optimizer = keras.optimizers.SGD(learning_rate=0.001,momentum=0.9)

Nesterov Accelerated Gradient

 The idea of Nesterov
Momentum optimization, or Nesterov Accelerated Gradient (NAG), is to measure the gradient of the cost function not at the local position but slightly ahead in the direction of the momentum

m -> βm − η∇θJ(θ + βm)

In [14]:
optimizer = keras.optimizers.SGD(learning_rate=.001, momentum=0.9, nesterov = True)

AdaGrad

Gradient Descent starts by quickly going
down the steepest slope, then slowly goes down the bottom of the valley. It would be
nice if the algorithm could detect this early on and correct its direction to point a bit
more toward the global optimum.<br>
The AdaGrad algorithm achieves this by scaling down the gradient vector along the
steepest dimensions 

This is super advanced for me to learn now.

RMSProp

Although AdaGrad slows down a bit too fast and ends up never converging to the
global optimum, the RMSProp algorithm fixes this by accumulating only the gradients from the most recent iterations (as opposed to all the gradients since the beginning of training). It does so by using exponential decay in the first step

<big>Adam and Nadam Optimization</big>


Adam, which stands for adaptive moment estimation, combines the ideas of Momentum optimization and RMSProp: just like Momentum optimization it keeps track of
an exponentially decaying average of past gradients, and just like RMSProp it keeps
track of an exponentially decaying average of past squared gradients

In [15]:
optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1 = 0.9, beta_2 = 0.999)

Adamax and Nadam optimization

Learning Rate Scheduling

<big>Avoiding Overfitting Through Regularization</big>

L1 and L2 Regularization

In [16]:
layer = keras.layers.Dense(100, activation='elu',
                           kernel_initializer = "he_normal",
                           kernel_regularizer=keras.regularizers.l2(0.01))

Dropout

In [17]:
model = keras.models.Sequential([
 keras.layers.Flatten(input_shape=[28, 28]),
 keras.layers.Dropout(rate=0.2),
 keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
 keras.layers.Dropout(rate=0.2),
 keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
 keras.layers.Dropout(rate=0.2),
 keras.layers.Dense(10, activation="softmax")
])


Monte-Carlo (MC) Dropout

Max-norm Regularization

In [19]:
keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal",
                   kernel_constraint=keras.constraints.max_norm(1.))

<Dense name=dense_13, built=False>